In [1]:
# ==========================================
# CELL 0: Install Dependencies
# ==========================================
!pip install -q "numpy<2" pypdf chromadb sentence-transformers langchain-community langchain-huggingface


In [2]:
!pip3 install "numpy<2.0" "scikit-learn<1.5.0" --force-reinstall


Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp39-cp39-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached scikit_learn-1.4.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
Using cached numpy-1.26.4-cp39-cp39-macosx_11_0_arm64.whl (14.0 MB)
Using cached scikit_learn-1.4.2-cp39-cp39-macosx_12_0_arm64.whl (10.5 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl (30.3 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.7.0
    Uninstalling threadpoolctl-3.7.0:
      Successfully uninstalled threadpoolctl-3.7.00m 0/5 [threadpoolctl]
  Attempting uninstall: numpy━━

In [ ]:
# ==========================================
# CELL 1: Environment & Token Setup
# ==========================================
import os

# Set your Hugging Face API Token
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"
# Replace with your active HF Token

os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
os.environ["HF_TOKEN"] = HF_TOKEN

print("1. Setup completed successfully! ⚙️")


1. Setup completed successfully! ⚙️


In [4]:
# ==========================================
# CELL 2: PDF Document Loading & Chunking
# ==========================================
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Specify PDF Path
pdf_path = "Sami_Ercetingoz_Thesis.pdf"

if not os.path.exists(pdf_path):
    print(f"❌ File not found at '{pdf_path}'. Current working directory: {os.getcwd()}")
else:
    # 1. Load PDF
    print(f"Loading document: {pdf_path}...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Total pages loaded: {len(documents)}")

    # 2. Split Document into Chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Successfully split into {len(chunks)} text chunks.")


/Users/kaanercetingoz/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/kaanercetingoz/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading document: Sami_Ercetingoz_Thesis.pdf...
Total pages loaded: 33
Successfully split into 81 text chunks.


In [5]:
# ==========================================
# CELL 3: Vector Store & Embeddings Setup
# ==========================================
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize Hugging Face Embedding Model (Runs locally)
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Create Vector Store from Document Chunks
print("Creating vector database from document chunks...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector database successfully created and indexed in Chroma! 🚀")


Loading embedding model...
Creating vector database from document chunks...
Vector database successfully created and indexed in Chroma! 🚀


In [6]:
# ==========================================
# CELL 4: Conversational RAG Pipeline & LLM
# ==========================================
import os
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize LLM via Hugging Face Serverless API
raw_llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-3B-Instruct",
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.2,
    max_new_tokens=512
)
llm = ChatHuggingFace(llm=raw_llm)

# 2. Setup Vector Store Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# 3. Session History Storage
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 4. System Prompt Design
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert AI research assistant analyzing a master's thesis.

STRICT RULES:
1. Respond ONLY in English.
2. Base your answers strictly on the provided context and conversation history.
3. If the answer cannot be found in the context, clearly state that the information is not available in the provided text. Do not make up facts.

Context:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 5. RAG Chain Construction
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def combine_inputs(inputs):
    docs = retriever.invoke(inputs["question"])
    return {
        "context": format_docs(docs),
        "question": inputs["question"],
        "chat_history": inputs.get("chat_history", [])
    }

chain = combine_inputs | qa_prompt | llm | StrOutputParser()

# 6. Runnable Chain with Memory
conversational_rag_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("Conversational RAG Assistant initialized with Llama-3.2! 🚀")


Conversational RAG Assistant initialized with Llama-3.2! 🚀


/Users/kaanercetingoz/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
# ==========================================
# CELL 5: Testing Question Answering & Memory
# ==========================================
config = {"configurable": {"session_id": "user_session_1"}}

# --- Question 1 ---
print("--- QUESTION 1 ---")
query1 = "What is the main objective and scope of the SOC-Sentinel thesis?"
print(f"USER: {query1}\n")

response1 = conversational_rag_chain.invoke(
    {"question": query1},
    config=config
)
print(f"ASSISTANT:\n{response1}")

print("\n" + "="*60 + "\n")

# --- Question 2 (Memory Test) ---
print("--- QUESTION 2 (MEMORY TEST) ---")
query2 = "What machine learning models or reinforcement learning algorithms were used in this system?"
print(f"USER: {query2}\n")

response2 = conversational_rag_chain.invoke(
    {"question": query2},
    config=config
)
print(f"ASSISTANT:\n{response2}")


--- QUESTION 1 ---
USER: What is the main objective and scope of the SOC-Sentinel thesis?

ASSISTANT:
The main objective and scope of the SOC-Sentinel thesis can be found in Chapter 1, Section 1.3. According to the text, the thesis objective is to develop an autonomous decision-making system, SOC-Sentinel, that can proactively detect and respond to security threats in real-time, without human intervention. The scope of the thesis is to design, develop, and evaluate the performance of the SOC-Sentinel system, which is based on a Dueling Double Deep Q-Network (D3QN) algorithm.


--- QUESTION 2 (MEMORY TEST) ---
USER: What machine learning models or reinforcement learning algorithms were used in this system?

ASSISTANT:
The SOC-Sentinel system uses a Dueling Double Deep Q-Network (D3QN) algorithm, which is a type of reinforcement learning model. This model is based on Deep Q-Networks (DQN) and combines the advantages of Dueling Networks, which were introduced in [6] (Deep reinforcement le